In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

pdata2021 = Path(r"C:\Users\alys_\Downloads\pdata2021")

In [9]:
br = pd.read_csv(
    pdata2021 / "building_res.txt",
    sep="\t",
    dtype=str,
    usecols=res_cols,
    low_memory=False,
    encoding="cp1252"
)

br["source"] = "building_res"

In [10]:
other_cols = [
    "acct",
    "property_use_cd",
    "bld_num",
    "impr_tp",
    "impr_mdl_cd",
    "structure",
    "qa_cd",
    "category",
    "pgi_dscr",
    "prop_nm",
    "units"
]

bo = pd.read_csv(
    pdata2021 / "building_other.txt",
    sep="\t",
    dtype=str,
    usecols=other_cols,
    low_memory=False,
    encoding="cp1252"
)

bo["source"] = "building_other"

In [11]:
print(br.shape)
print(bo.shape)

print(br.head())
print(bo.head())

(1226615, 7)
(153908, 12)
                        acct bld_num impr_tp impr_mdl_cd structure qa_cd  \
0  0020720000014                   1    1001        101        R      B    
1  0021440000001                   1    1001        101        R      B    
2  0021440000003                   1    1002        102        R      B    
3  0021440000008                   1    1001        101        R      D    
4  0021440000008                   2    1001        101        R      D    

         source  
0  building_res  
1  building_res  
2  building_res  
3  building_res  
4  building_res  
                        acct property_use_cd bld_num impr_tp impr_mdl_cd  \
0  0010020000016                          F1       1    4321        8350   
1  0010040000001                          X1       1    4398        8454   
2  0010100000001                          F1       1    4353        8344   
3  0010120000010                          X1       1    4670        8335   
4  0010130000002             

In [12]:
def clean_code(series):
    return (
        series
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

for df in [br, bo]:
    for col in df.columns:
        if col != "source":
            df[col] = clean_code(df[col])

In [13]:
print(br["acct"].head().tolist())
print(bo["acct"].head().tolist())

['0020720000014', '0021440000001', '0021440000003', '0021440000008', '0021440000008']
['0010020000016', '0010040000001', '0010100000001', '0010120000010', '0010130000002']


In [27]:
# Columns that building_other has but building_res does not
for col in ["property_use_cd", "category", "pgi_dscr", "prop_nm", "units"]:
    br[col] = pd.NA

building_cols = [
    "acct",
    "bld_num",
    "property_use_cd",
    "impr_tp",
    "impr_mdl_cd",
    "structure",
    "qa_cd",
    "category",
    "pgi_dscr",
    "prop_nm",
    "units",
    "source"
]

br = br[building_cols]
bo = bo[building_cols]

buildings = pd.concat(
    [br, bo],
    ignore_index=True
)

print(buildings.shape)

(1380523, 12)


In [28]:
CORE_BUILDING_TYPES = {
    "1002",  # Duplex
    "1003",  # Triplex
    "1004",  # Fourplex
    "4209",  # Apartment 4-20 units
    "4211",  # Garden apartment
    "4212",  # Mid-rise apartment
    "4213",  # Manufactured housing park
    "4214",  # High-rise apartment
    "4221",  # Subsidized housing
    "4222",  # Tax-credit apartment
    "4299",  # Apartment structure
}

CONDITIONAL_BUILDING_TYPES = {
    "1001",  # Single-family
    "1005",  # Mixed residential/commercial
    "1006",  # Condo
    "1007",  # Townhome
    "1008",  # Mobile home
    "4201",  # Residential structure
    "4319",  # Commercial building - mixed residential
}

REVIEW_BUILDING_TYPES = {
    "1025",  # Farm single-family dwelling
    "4313",  # Dormitory
    "4316",  # Nursing home
    "4317",  # Retirement home
    "4318",  # Boarding/rooming house
    "4320",  # Extended stay
}

In [29]:
CORE_PGI = {
    "13",  # Manufactured housing parks
    "16",  # Apartments mid/high rise
    "17",  # Garden apartments
    "18",  # Tax-credit apartments
    "19",  # Subsidized apartments
}

CONDITIONAL_PGI = {
    "30",  # Condos
}

In [30]:
CORE_STATE_CLASSES = {
    "B1",
    "B2",
    "B3",
    "B4",
}

CONDITIONAL_STATE_CLASSES = {
    "A1",
    "A2",
    "A4",
    "Z1",
    "Z2",
    "Z3",
    "Z4",
    "Z5",
}

In [31]:
buildings["explicit_rental_building"] = (
    buildings["impr_tp"].isin(CORE_BUILDING_TYPES)
)

buildings["explicit_rental_state"] = (
    buildings["property_use_cd"].isin(CORE_STATE_CLASSES)
)

buildings["explicit_rental_pgi"] = (
    buildings["category"].isin(CORE_PGI)
)

buildings["explicit_rental"] = (
    buildings["explicit_rental_building"]
    | buildings["explicit_rental_state"]
    | buildings["explicit_rental_pgi"]
)

In [32]:
buildings["explicit_rental"] = (
    buildings["explicit_rental_building"]
    | buildings["explicit_rental_state"]
    | buildings["explicit_rental_pgi"]
)

In [34]:
lambda x: (x == "building_res").any()
lambda x: (x == "building_other").any()

<function __main__.<lambda>(x)>

In [35]:
# Create source flags BEFORE grouping
buildings["in_building_res"] = (
    buildings["source"] == "building_res"
).astype("uint8")

buildings["in_building_other"] = (
    buildings["source"] == "building_other"
).astype("uint8")

# Convert rental evidence flags to compact integers too
flag_cols = [
    "explicit_rental",
    "explicit_rental_building",
    "explicit_rental_state",
    "explicit_rental_pgi"
]

for col in flag_cols:
    buildings[col] = buildings[col].astype("uint8")

# Now aggregate using only fast built-in operations
rental_building_evidence = (
    buildings[
        [
            "acct",
            "explicit_rental",
            "explicit_rental_building",
            "explicit_rental_state",
            "explicit_rental_pgi",
            "in_building_res",
            "in_building_other"
        ]
    ]
    .groupby("acct", sort=False, as_index=False)
    .max()
)

In [36]:
print(rental_building_evidence.shape)

print(
    "Duplicate accounts:",
    rental_building_evidence["acct"].duplicated().sum()
)

print(
    rental_building_evidence[
        [
            "explicit_rental",
            "explicit_rental_building",
            "explicit_rental_state",
            "explicit_rental_pgi",
            "in_building_res",
            "in_building_other"
        ]
    ].sum()
)

(1264719, 7)
Duplicate accounts: 0
explicit_rental               16297
explicit_rental_building      16286
explicit_rental_state          5662
explicit_rental_pgi            2731
in_building_res             1197988
in_building_other             66731
dtype: uint64


In [38]:
print(
    rental_building_evidence["explicit_rental"]
    .value_counts(dropna=False)
)

print(
    rental_building_evidence["explicit_rental"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

explicit_rental
0    1248422
1      16297
Name: count, dtype: int64
explicit_rental
0    98.71
1     1.29
Name: proportion, dtype: float64


In [39]:
# Convert units to numeric
buildings["units_num"] = pd.to_numeric(
    buildings["units"],
    errors="coerce"
)

# Keep only accounts currently identified by explicit rental evidence
explicit_accts = set(
    rental_building_evidence.loc[
        rental_building_evidence["explicit_rental"] == 1,
        "acct"
    ]
)

rental_rows = buildings[
    buildings["acct"].isin(explicit_accts)
].copy()

print("Explicit rental accounts:", len(explicit_accts))
print("Rental building rows:", len(rental_rows))

print("\nRows with a usable HCAD units value:")
print(rental_rows["units_num"].notna().sum())

print("\nDistribution of units values:")
print(rental_rows["units_num"].describe())

Explicit rental accounts: 16297
Rental building rows: 60692

Rows with a usable HCAD units value:
45984

Distribution of units values:
count    45984.000000
mean       262.070742
std        221.418002
min          1.000000
25%        126.000000
50%        240.000000
75%        336.000000
max       1792.000000
Name: units_num, dtype: float64


In [40]:
units_account_max = (
    rental_rows
    .groupby("acct", as_index=False)
    .agg(
        max_reported_units=("units_num", "max")
    )
)

print(units_account_max["max_reported_units"].describe())
print(
    "Accounts with reported units:",
    units_account_max["max_reported_units"].notna().sum()
)

count    5984.000000
mean      102.993650
std       141.836144
min         1.000000
25%         5.000000
50%        18.500000
75%       190.000000
max      1792.000000
Name: max_reported_units, dtype: float64
Accounts with reported units: 5984


In [42]:
units_by_building = (
    rental_rows
    .groupby(["acct", "bld_num"], as_index=False)
    .agg(
        units_for_building=("units_num", "max")
    )
)

units_account_bldsum = (
    units_by_building
    .groupby("acct", as_index=False)
    .agg(
        sum_building_units=("units_for_building", "sum")
    )
)

In [44]:
unit_check = units_account_max.merge(
    units_account_bldsum,
    on="acct",
    how="outer",
    validate="one_to_one"
)

print(unit_check[
    ["max_reported_units", "sum_building_units"]
].describe())

print("\nTotal using max per account:")
print(unit_check["max_reported_units"].sum())

print("\nTotal using sum across buildings:")
print(unit_check["sum_building_units"].sum())

       max_reported_units  sum_building_units
count         5984.000000        16297.000000
mean           102.993650          731.587225
std            141.836144         4490.511483
min              1.000000            0.000000
25%              5.000000            0.000000
50%             18.500000            0.000000
75%            190.000000           10.000000
max           1792.000000       311808.000000

Total using max per account:
616314.0

Total using sum across buildings:
11922677.0


In [45]:
n_explicit = len(explicit_accts)

n_with_units = unit_check["max_reported_units"].notna().sum()

print("Explicit rental accounts:", n_explicit)
print("Accounts with HCAD units:", n_with_units)
print(
    "Unit-data coverage:",
    round(100 * n_with_units / n_explicit, 2),
    "%"
)

Explicit rental accounts: 16297
Accounts with HCAD units: 5984
Unit-data coverage: 36.72 %


In [46]:
SMALL_MF_UNITS = {
    "1002": 2,   # duplex
    "1003": 3,   # triplex
    "1004": 4,   # fourplex
}

rental_rows["units_from_type"] = (
    rental_rows["impr_tp"]
    .map(SMALL_MF_UNITS)
)

In [47]:
small_mf_units = (
    rental_rows
    .groupby("acct", as_index=False)
    .agg(
        units_from_type=("units_from_type", "max")
    )
)

unit_check = unit_check.merge(
    small_mf_units,
    on="acct",
    how="left",
    validate="one_to_one"
)

unit_check["estimated_units_prelim"] = (
    unit_check["max_reported_units"]
    .fillna(unit_check["units_from_type"])
)

In [48]:
print(
    "Accounts with preliminary unit estimate:",
    unit_check["estimated_units_prelim"].notna().sum()
)

print(
    "Coverage:",
    round(
        100 *
        unit_check["estimated_units_prelim"].notna().sum()
        / len(explicit_accts),
        2
    ),
    "%"
)

print(
    "Preliminary known rental units:",
    unit_check["estimated_units_prelim"].sum()
)

Accounts with preliminary unit estimate: 15978
Coverage: 98.04 %
Preliminary known rental units: 637393.0


In [49]:
exempt = pd.read_csv(
    pdata2021 / "jur_exempt_cd.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252",
    low_memory=False
)

print(exempt.shape)
print(exempt.head(10))
print(exempt.columns.tolist())

(1507911, 2)
            acct exempt_cat
0  0010010000013        TOT
1  0010020000001           
2  0010020000003           
3  0010020000004           
4  0010020000013        TOT
5  0010020000015        TOT
6  0010020000016           
7  0010020000023        TOT
8  0010020000024           
9  0010030000001        TOT
['acct', 'exempt_cat']


In [50]:
for col in exempt.columns:
    exempt[col] = (
        exempt[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )
    

In [51]:
print(
    exempt["exempt_cat"]
    .value_counts(dropna=False)
    .head(40)
)


exempt_cat
RES            797823
<NA>           597509
TOT             82319
RES V14          6685
RES VTX          4034
PAR              3211
RES V14 VTX      3075
RES SOL          2188
RES V13          1637
RES V11          1494
RES V12          1157
PRO               970
APR               899
V14               494
POL               471
VTX               442
RES STX           312
RES VS4           260
D11 RES           234
RES STX VS4       230
RES V13 VTX       222
SOL               166
D12 RES           153
D11               116
HIS RES           111
V13               108
V11               107
PEX                99
D13                99
APD APR            85
RES V12 VTX        79
PRO RES            65
V12                63
RES VS1            63
D12                52
ABT                48
RES SOL V14        45
RES STT            41
RES VS2            40
RES V11 VTX        39
Name: count, dtype: int64[pyarrow]


In [52]:
print("Rows:", len(exempt))
print("Unique accounts:", exempt["acct"].nunique())

print(
    exempt.groupby("acct")
    .size()
    .describe()
)

Rows: 1507911
Unique accounts: 1507911
count    1507911.0
mean           1.0
std            0.0
min            1.0
25%            1.0
50%            1.0
75%            1.0
max            1.0
dtype: float64


In [53]:
exemption_evidence = exempt[
    ["acct", "exempt_cat"]
].copy()

# Add spaces around the list so exact codes are easy to detect
exemption_evidence["_exempt_tokens"] = (
    " "
    + exemption_evidence["exempt_cat"].fillna("")
    + " "
)

exemption_evidence["has_res_homestead"] = (
    exemption_evidence["_exempt_tokens"]
    .str.contains(" RES ", regex=False)
    .astype("uint8")
)

exemption_evidence["has_partial_homestead"] = (
    exemption_evidence["_exempt_tokens"]
    .str.contains(" PAR ", regex=False)
    .astype("uint8")
)

exemption_evidence["has_apportioned_residential"] = (
    exemption_evidence["_exempt_tokens"]
    .str.contains(" APR ", regex=False)
    .astype("uint8")
)

exemption_evidence["has_low_income_housing"] = (
    exemption_evidence["_exempt_tokens"]
    .str.contains(" LIH ", regex=False)
    .astype("uint8")
)

exemption_evidence = exemption_evidence.drop(
    columns="_exempt_tokens"
)

In [54]:
print(
    exemption_evidence[
        [
            "has_res_homestead",
            "has_partial_homestead",
            "has_apportioned_residential",
            "has_low_income_housing"
        ]
    ].sum()
)

print(
    "Duplicate accounts:",
    exemption_evidence["acct"].duplicated().sum()
)

has_res_homestead              820507
has_partial_homestead            3248
has_apportioned_residential      1021
has_low_income_housing              2
dtype: uint64
Duplicate accounts: 0


In [56]:
exemption_evidence["any_homestead_evidence"] = (
    (
        exemption_evidence["has_res_homestead"]
        | exemption_evidence["has_partial_homestead"]
        | exemption_evidence["has_apportioned_residential"]
    )
    .astype("uint8")
)

In [57]:
print(
    exemption_evidence["any_homestead_evidence"]
    .value_counts()
)

print(
    exemption_evidence["any_homestead_evidence"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

any_homestead_evidence
1    824769
0    683142
Name: count, dtype: int64
any_homestead_evidence
1    54.7
0    45.3
Name: proportion, dtype: float64


In [58]:
print(
    exemption_evidence[
        [
            "has_res_homestead",
            "has_partial_homestead",
            "has_apportioned_residential",
            "has_low_income_housing",
            "any_homestead_evidence"
        ]
    ].sum()
)

has_res_homestead              820507
has_partial_homestead            3248
has_apportioned_residential      1021
has_low_income_housing              2
any_homestead_evidence         824769
dtype: uint64


In [59]:
owners_header = pd.read_csv(
    pdata2021 / "owners.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252",
    nrows=0
)

print(owners_header.columns.tolist())

['acct', 'ln_num', 'name', 'aka', 'pct_own']


In [60]:
owners = pd.read_csv(
    pdata2021 / "owners.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252",
    low_memory=False
)

print("Rows:", len(owners))
print("Unique accounts:", owners["acct"].nunique())
print(owners.head())

Rows: 1735161
Unique accounts: 1507914
            acct ln_num                        name aka pct_own
0  0010010000013      1             CITY OF HOUSTON      1.0000
1  0010020000001      1               CURRENT OWNER      1.0000
2  0010020000003      1  MILBY CHARLES FAMILY PTNSH      1.0000
3  0010020000004      1   MICHAEL RYAN FEAGIN TRUST      1.0000
4  0010020000013      1   BUFFALO BAYOU PARTNERSHIP      1.0000


In [61]:
for col in owners.columns:
    owners[col] = (
        owners[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

In [62]:
owner_counts = owners.groupby("acct").size()

print(owner_counts.describe())

print(
    "Accounts with more than one owner:",
    (owner_counts > 1).sum()
)

count    1.507914e+06
mean     1.150703e+00
std      3.714373e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.100000e+01
dtype: float64
Accounts with more than one owner: 221383


In [63]:
print(
    owner_counts
    .sort_values(ascending=False)
    .head(20)
)

acct
0915520000458    11
0710900110205    10
0630740720006    10
0740550000078    10
0660220000433     9
0964390000014     9
0390700000016     9
0543100000017     8
0761950070077     8
0691060130008     8
0852800000020     8
0986350000199     8
1126090000001     7
0760870010004     7
0771020130014     7
0752380060015     7
0162650440013     7
0551790000022     7
0640490020008     7
0710900150313     7
dtype: int64


In [64]:
owners["pct_own_num"] = pd.to_numeric(
    owners["pct_own"],
    errors="coerce"
)

In [65]:
pct_check = (
    owners
    .groupby("acct", as_index=False)
    .agg(
        owner_count=("name", "size"),
        pct_own_sum=("pct_own_num", "sum"),
        max_pct_own=("pct_own_num", "max")
    )
)

print(pct_check["pct_own_sum"].describe())

print(
    pct_check["pct_own_sum"]
    .round(3)
    .value_counts()
    .head(20)
)

count    1507914.0
mean      0.998113
std        0.04107
min            0.0
25%            1.0
50%            1.0
75%            1.0
max            6.3
Name: pct_own_sum, dtype: Float64
pct_own_sum
1.0      1502884
0.5         3076
0.0         1347
2.0          144
0.99          80
0.1           59
1.5           38
0.34          32
1.01          28
0.999         23
0.33          20
0.25          15
0.333         11
0.55           8
0.01           7
0.75           7
0.993          7
0.37           7
0.8            6
0.67           6
Name: count, dtype: Int64


In [69]:
CORPORATE_PATTERN = (
    r"\b(?:"
    r"LLC|L\.L\.C\.|"
    r"LLP|L\.L\.P\.|"
    r"LP|L\.P\.|"
    r"INC|INCORPORATED|"
    r"CORP|CORPORATION|"
    r"LTD|LIMITED|"
    r"PARTNERSHIP|PTNSH"
    r")\b"
)

owners["is_corporate_owner"] = (
    owners["name"]
    .fillna("")
    .str.contains(CORPORATE_PATTERN, case=False, regex=True)
    .astype("uint8")
)

In [70]:
owners["corporate_pct"] = (
    owners["pct_own_num"]
    * owners["is_corporate_owner"]
)

In [71]:
owner_evidence = (
    owners
    .groupby("acct", sort=False, as_index=False)
    .agg(
        owner_count=("name", "size"),
        corporate_ownership_share=("corporate_pct", "sum"),
        max_owner_share=("pct_own_num", "max")
    )
)

In [72]:
owner_evidence["has_corporate_owner"] = (
    owner_evidence["corporate_ownership_share"] > 0
).astype("uint8")

owner_evidence["majority_corporate_owned"] = (
    owner_evidence["corporate_ownership_share"] >= 0.50
).astype("uint8")

In [73]:
print(owner_evidence.shape)

print(
    owner_evidence[
        [
            "owner_count",
            "corporate_ownership_share",
            "max_owner_share"
        ]
    ].describe()
)

print("\nCorporate owner flag:")
print(
    owner_evidence["has_corporate_owner"]
    .value_counts()
)

print("\nMajority corporate-owned:")
print(
    owner_evidence["majority_corporate_owned"]
    .value_counts()
)

(1507914, 6)
       owner_count  corporate_ownership_share  max_owner_share
count    1507914.0                  1507914.0        1507914.0
mean      1.150703                    0.11057         0.963482
std       0.371437                    0.31333          0.13296
min            1.0                        0.0              0.0
25%            1.0                        0.0              1.0
50%            1.0                        0.0              1.0
75%            1.0                        0.0              1.0
max           11.0                       3.25              5.8

Corporate owner flag:
has_corporate_owner
0    1340616
1     167298
Name: count, dtype: int64

Majority corporate-owned:
majority_corporate_owned
0    1340734
1     167180
Name: count, dtype: int64


In [74]:
print(
    owners.loc[
        owners["is_corporate_owner"] == 1,
        ["acct", "name", "pct_own_num"]
    ].sample(20, random_state=42)
)

                  acct                                    name  pct_own_num
134429   0410290040169                     GATX TERMINALS CORP          1.0
315396   0670470040154                     BRINKMAN CENTER LLC          1.0
168253   0440210000030                   CASNAR PROPERTIES LLC          1.0
554270   0903280000003               SILVERWOOD BUILDERS I INC          1.0
899720   1135030000001                    IBNI INVESTMENTS LLC          1.0
555121   0903650000013                     LAMA FOUNDATION INC          0.0
1504444  1309270010015                         ASSOCIATION INC          0.0
1569647  1357850010001             GEOKEN INVESTMENT GROUP LLC          1.0
551740   0901690000002                           HYDRO SPY LLC          1.0
1652324  1400790090012                CYPRESS CREEK WAYSIDE LP          1.0
1050311  1155600050034                         LUCKY TWINS LLC          1.0
1402568  1270910000239                  POST OAK LOFT 3217 LLC          1.0
527997   086

In [75]:
weird_pct = pct_check[
    (pct_check["pct_own_sum"] < 0.99)
    | (pct_check["pct_own_sum"] > 1.01)
]

print("Accounts with unusual ownership totals:", len(weird_pct))

print(
    weird_pct["pct_own_sum"]
    .value_counts()
    .sort_index()
    .head(30)
)

Accounts with unusual ownership totals: 4872
pct_own_sum
0.0       1347
0.001        1
0.0024       1
0.005        1
0.0098       1
0.01         6
0.03         1
0.0811       1
0.083        1
0.1         59
0.11         1
0.12         1
0.125        1
0.13         1
0.14         1
0.16         1
0.195        1
0.2          6
0.2222       1
0.25        15
0.33        20
0.333        1
0.3333       2
0.3334       8
0.334        1
0.34        32
0.35         1
0.37         7
0.375        1
0.4          1
Name: count, dtype: Int64


In [76]:
weird_high = weird_pct.loc[
    weird_pct["pct_own_sum"] > 1.01,
    "acct"
].head(10)

print(
    owners[
        owners["acct"].isin(weird_high)
    ][
        ["acct", "ln_num", "name", "aka", "pct_own", "pct_own_num"]
    ].sort_values(["acct", "ln_num"])
)

                acct ln_num                              name  \
109    0010270000001      1          UNITED STATES OF AMERICA   
110    0010270000001      2              PEMANENT SCHOOL FUND   
2033   0031440170002      1                      PLATT C DAER   
2034   0031440170002      2                     PLATT PAULA D   
2035   0031440170002      3                    PLATT JUDITH R   
4793   0041730000066      1                VALENZUELA ERNESTO   
4794   0041730000066      2               VALENZUELA HERLINDA   
4916   0041730000223      1                  COUNTY OF HARRIS   
4917   0041730000223      2        PARCEL 34 (ALDINE MAIL RD)   
13792  0092610000007      1           AHUATZI-SALDANA IGNACIO   
13793  0092610000007      2                      LOPEZ ELVIRA   
19223  0120740000009      1                   CAZARES JESUS A   
19224  0120740000009      2                      MAYNEZ NORMA   
28311  0150360000013      1           METRO TRANSIT AUTHORITY   
28312  0150360000013     

In [77]:
weird_low = weird_pct.loc[
    weird_pct["pct_own_sum"] < 0.99,
    "acct"
].head(10)

print(
    owners[
        owners["acct"].isin(weird_low)
    ][
        ["acct", "ln_num", "name", "aka", "pct_own", "pct_own_num"]
    ].sort_values(["acct", "ln_num"])
)

               acct ln_num                                  name   aka  \
2240  0031670000002      1                         CURRENT OWNER  <NA>   
2612  0031960000038      1                         SOLIS EUFEMIA  <NA>   
2939  0032180000008      1        COPPERTOP OPPORTUNITY ZONE INC  <NA>   
3104  0032240000019      1                         CURRENT OWNER  <NA>   
3759  0040360000002      1                     CASTELLO BEATRICE  <NA>   
4723  0041710000140      1                          RAMOS JAVIER  <NA>   
4724  0041710000140      2                         RAMOS VANESSA  <NA>   
4850  0041730000129      1                   GARCIA ENRIQUE M JR  <NA>   
5430  0041790000250      1            CARDENAS REAL PROPERTY LLC  <NA>   
5813  0042070180010      1                         CURRENT OWNER  <NA>   
6095  0050110000007      1  AMBASSADORIAL ICH REALTY PARNERS LLC  <NA>   

     pct_own  pct_own_num  
2240  0.0000          0.0  
2612  0.3334       0.3334  
2939  0.5000          0.5  

In [78]:
owner_evidence_simple = (
    owners
    .groupby("acct", sort=False, as_index=False)
    .agg(
        has_corporate_owner=("is_corporate_owner", "max")
    )
)

In [79]:
print(owner_evidence_simple["has_corporate_owner"].value_counts())
print("Duplicates:", owner_evidence_simple["acct"].duplicated().sum())

has_corporate_owner
0    1335754
1     172160
Name: count, dtype: int64
Duplicates: 0


In [80]:
real_cols = [
    "acct",
    "yr",
    "mailto",

    "mail_addr_1",
    "mail_addr_2",
    "mail_city",
    "mail_state",
    "mail_zip",

    "site_addr_1",
    "site_addr_2",
    "site_addr_3",

    "state_class",

    "Neighborhood_Code",
    "Neighborhood_Grp",
    "Market_Area_1",
    "Market_Area_1_Dscr",

    "yr_impr",
    "bld_ar",
    "land_ar",
    "acreage",

    "land_val",
    "bld_val",
    "tot_appr_val",
    "tot_mkt_val",

    "new_own_dt"
]

real = pd.read_csv(
    pdata2021 / "real_acct.txt",
    sep="\t",
    dtype=str,
    usecols=real_cols,
    encoding="cp1252",
    low_memory=False
)

In [81]:
for col in real.columns:
    real[col] = (
        real[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

In [82]:
print(real.shape)
print("Unique accounts:", real["acct"].nunique())
print("Duplicate accounts:", real["acct"].duplicated().sum())

print(
    real["state_class"]
    .value_counts(dropna=False)
    .head(30)
)

(1507914, 25)
Unique accounts: 1507914
Duplicate accounts: 0
state_class
A1      1078901
X1        71262
F1        64766
C1        54723
C3        34736
M3        29418
C2        28855
Z4        27829
Z3        13643
O1        13400
Z1        11819
A2        11078
A3        11004
Z5        10265
X3         8224
B2         7272
B1         5700
1D1        4842
O2         3608
D2         2825
A4         2784
F2         2449
J3         1875
X2         1554
J5         1166
Z0          759
XJ          539
B3          510
E1          468
TMBR        434
Name: count, dtype: int64[pyarrow]


In [83]:
fixtures = pd.read_csv(
    pdata2021 / "fixtures.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252",
    low_memory=False
)

for col in fixtures.columns:
    fixtures[col] = (
        fixtures[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

fixtures["units_num"] = pd.to_numeric(
    fixtures["units"],
    errors="coerce"
)

print(fixtures.shape)
print(fixtures.head())
print(fixtures.columns.tolist())

(7573519, 6)
            acct bld_num type                type_dscr   units  units_num
0  0010020000016       1  EL3     Elev:  Hydro / Frght    1.00        1.0
1  0010020000016       1  INT  Interior Finish Percent  100.00      100.0
2  0010020000016       1  WHT              Wall Height   12.00       12.0
3  0010040000001       1  INT  Interior Finish Percent  100.00      100.0
4  0010040000001       1  OD2     OH Door:  Roll Steel    2.00        2.0
['acct', 'bld_num', 'type', 'type_dscr', 'units', 'units_num']


In [84]:
print(
    fixtures["type"]
    .value_counts()
    .head(50)
)

type
RMB     1229543
RMT     1229344
RMF     1229340
STY     1228450
FXA      596409
RMH      495925
RMR      439013
FPM      393415
FPW      220853
INT      143881
WHT      100154
WH        57857
FPD       35726
MAS       35206
OD1       31262
OWR       17584
STC       14995
AP2       11147
AP1       10511
OD2        9162
SF2        6113
FPO        4589
REL        4479
EL2        3366
FRC        3263
AP3        2902
TBVT       2679
TBVM       1622
ATR        1537
EL1        1485
EL4        1226
SS1        1004
TBVS        963
AP0         850
UNT         632
NRA         557
OD3         502
PRK         443
SF1         414
CW2         413
BC1         401
WHI         396
CW1         378
FL1         370
SPA         304
OD4         303
TBVL        275
BE1         257
CW4         244
EL3         233
Name: count, dtype: int64[pyarrow]


In [85]:
feature_types = [
    "RMB",  # bedrooms
    "RMF",  # full baths
    "RMH",  # half baths
    "RMR",  # recreation rooms
    "RMT",  # total rooms
    "STC",  # stories
    "PRK",  # parking spaces
    "UNT",  # apartment units
    "AC1",  # central AC
    "AC2",  # unit AC
    "FL1",
    "FL2",
    "FL3",
    "FPD",
    "FPM",
    "FPO",
    "FPW",
]

print(
    fixtures[
        fixtures["type"].isin(feature_types)
    ][
        ["acct", "bld_num", "type", "type_dscr", "units_num"]
    ].head(50)
)

              acct bld_num type                  type_dscr  units_num
29   0010150000016       1  STC                  # Stories        3.0
39   0010160000014       1  STC                  # Stories        1.0
42   0010170000001       1  STC                  # Stories        2.0
62   0010190000003       1  STC                  # Stories        2.0
82   0010200000001       1  STC                  # Stories       12.0
123  0010280000012       1  STC                  # Stories        3.0
137  0010290000001       1  STC                  # Stories        4.0
138  0010290000001       1  PRK             Parking Spaces      435.0
150  0010320000004       1  STC                  # Stories        3.0
155  0010320000005       1  STC                  # Stories        3.0
159  0010320000008       1  STC                  # Stories        3.0
169  0010330000001       1  STC                  # Stories        3.0
172  0010330000004       1  STC                  # Stories        3.0
176  0010330000005  

In [86]:
fixture_dup_check = (
    fixtures
    .groupby(["acct", "bld_num", "type"])
    .size()
)

print(fixture_dup_check.describe())

print(
    fixture_dup_check[
        fixture_dup_check > 1
    ]
    .sort_values(ascending=False)
    .head(30)
)

count    7.534543e+06
mean     1.005171e+00
std      1.632495e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.910000e+02
dtype: float64
acct           bld_num  type
1289820000012  1        STC     191
1289820000025  1        STC     186
1289820000024  1        STC     179
1289820000011  1        STC     161
1155910140001  1        FL1      72
1155910140002  1        FL1      54
0342030020162  2        WH       15
0382760000005  2        WH       15
0641810000018  1        INT      14
                        WH       14
1210520010002  1        WH       12
0281730000006  10       WH       12
0402630000014  1        INT      12
1291390010001  1        WH       12
0421380000021  1        WH       12
0451800010012  1        INT      11
0560120160003  2        WH       11
0041750000168  1        WH       11
0410150070009  1        WH       11
0680930030002  1        WH       11
0611980000001  2        WH       11
               1        

In [87]:
print(
    "Unique accounts:",
    fixtures["acct"].nunique()
)

print(
    "Unique account-building pairs:",
    fixtures[["acct", "bld_num"]]
    .drop_duplicates()
    .shape[0]
)

Unique accounts: 1267694
Unique account-building pairs: 1370839


In [88]:
extra = pd.read_csv(
    pdata2021 / "extra_features.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252",
    low_memory=False
)

for col in extra.columns:
    extra[col] = (
        extra[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

print(extra.shape)
print(extra.head())
print(extra.columns.tolist())

(1042995, 11)
            acct bld_num count grade    cd   s_dscr                l_dscr cat  \
0  0010020000001       0     1     4  CPA1   PavAsp      Paving - Asphalt  MS   
1  0010020000013       0     1     4  CPA1   PavAsp      Paving - Asphalt  MS   
2  0010020000015       0     1     4  CPA1   PavAsp      Paving - Asphalt  MS   
3  0010020000016       1     1     4  CCP6   CpRfSl  CANOPY ROOF AND SLAB  MS   
4  0010040000001       1     5     4  CEN5  Enclos5    Enclosure,  Retail  MS   

            dscr          note      uts  
0  Miscellaneous  NEW FOR 2018  5000.00  
1  Miscellaneous          <NA>  3000.00  
2  Miscellaneous          <NA>  1250.00  
3  Miscellaneous          <NA>  1504.00  
4  Miscellaneous          <NA>  5880.00  
['acct', 'bld_num', 'count', 'grade', 'cd', 's_dscr', 'l_dscr', 'cat', 'dscr', 'note', 'uts']


In [89]:
print(
    extra[
        ["cd", "s_dscr", "l_dscr", "cat", "dscr"]
    ]
    .drop_duplicates()
    .head(100)
)

        cd      s_dscr                                    l_dscr cat  \
0     CPA1      PavAsp                          Paving - Asphalt  MS   
3     CCP6      CpRfSl                      CANOPY ROOF AND SLAB  MS   
4     CEN5     Enclos5                        Enclosure,  Retail  MS   
5     CEN6     Enclos6                        Enclosure,  Office  MS   
6     CBS1     BsmtNFp  Basement,Commercial Stg Non Fire Proofed  MS   
...    ...         ...                                       ...  ..   
1061  RSP1   SOLAR PNL                  Solar Photovoltaic Panel  SO   
1075  RRSL        Slab                  Dwelling Foundation Only  MS   
1179  CEN7     Enclos7                    Enclosure, Living Area  MS   
1340  RRC3  RCarpt/Qtr           Res Carport with Quarters Above  CR   
1383  RRS6     MUTBLDG                  Utility Building - Metal  OB   

               dscr  
0     Miscellaneous  
3     Miscellaneous  
4     Miscellaneous  
5     Miscellaneous  
6     Miscellaneous  
...

In [90]:
print(
    extra["l_dscr"]
    .value_counts()
    .head(50)
)

l_dscr
Frame Detached Garage                       174075
Canopy - Residential                        109579
Frame Utility Shed                          107002
Gunite Pool                                  96167
Carport - Residential                        79098
Pool SPA with Heater                         49586
CANOPY ROOF AND SLAB                         39967
Paving - Heavy Concrete                      32882
Porch, Open                                  27255
Porch,Open Upper                             22698
Metal Utility Shed                           20181
Skirting Att to M/H                          20163
Foundation Repaired                          17846
CANOPY ONLY                                  15448
Paving - Asphalt                             14600
Enclosure,  Office                           14131
Brick or Stone Detached Garage               13755
Residential Other Gross Value                12881
Cracked Slab                                 12367
Paving - Light Concrete 

In [91]:
fixtures.shape
fixtures["type"].value_counts().head(50)
fixture_dup_check.describe()
fixture_dup_check[fixture_dup_check > 1].sort_values(ascending=False).head(30)

acct           bld_num  type
1289820000012  1        STC     191
1289820000025  1        STC     186
1289820000024  1        STC     179
1289820000011  1        STC     161
1155910140001  1        FL1      72
1155910140002  1        FL1      54
0342030020162  2        WH       15
0382760000005  2        WH       15
0641810000018  1        INT      14
                        WH       14
1210520010002  1        WH       12
0281730000006  10       WH       12
0402630000014  1        INT      12
1291390010001  1        WH       12
0421380000021  1        WH       12
0451800010012  1        INT      11
0560120160003  2        WH       11
0041750000168  1        WH       11
0410150070009  1        WH       11
0680930030002  1        WH       11
0611980000001  2        WH       11
               1        WH       10
0272560000134  1        INT      10
1184340010001  1        WHT      10
0641030000014  1        WH        9
0410660000027  1        WH        9
0451800010012  1        WH        9

In [92]:
extra.shape
extra["l_dscr"].value_counts().head(50)

l_dscr
Frame Detached Garage                       174075
Canopy - Residential                        109579
Frame Utility Shed                          107002
Gunite Pool                                  96167
Carport - Residential                        79098
Pool SPA with Heater                         49586
CANOPY ROOF AND SLAB                         39967
Paving - Heavy Concrete                      32882
Porch, Open                                  27255
Porch,Open Upper                             22698
Metal Utility Shed                           20181
Skirting Att to M/H                          20163
Foundation Repaired                          17846
CANOPY ONLY                                  15448
Paving - Asphalt                             14600
Enclosure,  Office                           14131
Brick or Stone Detached Garage               13755
Residential Other Gross Value                12881
Cracked Slab                                 12367
Paving - Light Concrete 

In [93]:
print(
    fixtures[
        (fixtures["acct"] == "1289820000012")
        & (fixtures["bld_num"] == "1")
        & (fixtures["type"] == "STC")
    ][["acct", "bld_num", "type", "type_dscr", "units_num"]]
)

print(
    fixtures[
        (fixtures["acct"] == "1155910140001")
        & (fixtures["bld_num"] == "1")
        & (fixtures["type"] == "FL1")
    ][["acct", "bld_num", "type", "type_dscr", "units_num"]]
)

                  acct bld_num type  type_dscr  units_num
6472510  1289820000012       1  STC  # Stories        1.0
6472511  1289820000012       1  STC  # Stories        1.0
6472512  1289820000012       1  STC  # Stories        1.0
6472513  1289820000012       1  STC  # Stories        1.0
6472514  1289820000012       1  STC  # Stories        1.0
...                ...     ...  ...        ...        ...
6472696  1289820000012       1  STC  # Stories        1.0
6472697  1289820000012       1  STC  # Stories        1.0
6472698  1289820000012       1  STC  # Stories        1.0
6472699  1289820000012       1  STC  # Stories        1.0
6472700  1289820000012       1  STC  # Stories        1.0

[191 rows x 5 columns]
                  acct bld_num type             type_dscr  units_num
4309407  1155910140001       1  FL1  Fireplace:  Open (1)        1.0
4309408  1155910140001       1  FL1  Fireplace:  Open (1)        1.0
4309409  1155910140001       1  FL1  Fireplace:  Open (1)        1.0
4309

In [94]:
for acct, bld, typ in [
    ("1289820000012", "1", "STC"),
    ("1155910140001", "1", "FL1")
]:
    x = fixtures[
        (fixtures["acct"] == acct)
        & (fixtures["bld_num"] == bld)
        & (fixtures["type"] == typ)
    ]

    print("\n", acct, typ)
    print(x["units_num"].value_counts(dropna=False).head(20))


 1289820000012 STC
units_num
1.0    191
Name: count, dtype: Int64

 1155910140001 FL1
units_num
1.0    72
Name: count, dtype: Int64


In [95]:
extra["feature_text"] = (
    extra["l_dscr"]
    .fillna("")
    .str.upper()
)

In [96]:
extra["has_pool"] = (
    extra["feature_text"].str.contains(
        "POOL",
        regex=False
    )
).astype("uint8")

extra["has_pool_spa"] = (
    extra["feature_text"].str.contains(
        "POOL SPA",
        regex=False
    )
).astype("uint8")

extra["has_garage"] = (
    extra["feature_text"].str.contains(
        "GARAGE",
        regex=False
    )
).astype("uint8")

extra["has_carport"] = (
    extra["feature_text"].str.contains(
        "CARPORT",
        regex=False
    )
).astype("uint8")

extra["has_shed"] = (
    extra["feature_text"].str.contains(
        "SHED",
        regex=False
    )
).astype("uint8")

extra["has_porch"] = (
    extra["feature_text"].str.contains(
        "PORCH",
        regex=False
    )
).astype("uint8")

extra["has_outdoor_kitchen"] = (
    extra["feature_text"].str.contains(
        "OUTDOOR KITCHEN",
        regex=False
    )
).astype("uint8")

extra["has_solar"] = (
    extra["feature_text"].str.contains(
        "SOLAR",
        regex=False
    )
).astype("uint8")

extra["has_foundation_repair"] = (
    extra["feature_text"].str.contains(
        "FOUNDATION REPAIRED",
        regex=False
    )
).astype("uint8")

extra["has_cracked_slab"] = (
    extra["feature_text"].str.contains(
        "CRACKED SLAB",
        regex=False
    )
).astype("uint8")

In [97]:
extra_feature_cols = [
    "has_pool",
    "has_pool_spa",
    "has_garage",
    "has_carport",
    "has_shed",
    "has_porch",
    "has_outdoor_kitchen",
    "has_solar",
    "has_foundation_repair",
    "has_cracked_slab"
]

extra_features_account = (
    extra[
        ["acct"] + extra_feature_cols
    ]
    .groupby("acct", as_index=False, sort=False)
    .max()
)

assert extra_features_account["acct"].duplicated().sum() == 0

In [98]:
print(extra_features_account.shape)

print(
    extra_features_account[
        extra_feature_cols
    ].sum().sort_values(ascending=False)
)

(538584, 11)
has_garage               196366
has_shed                 118161
has_pool                 106115
has_carport               76899
has_pool_spa              49540
has_foundation_repair     17822
has_cracked_slab          12341
has_outdoor_kitchen       10774
has_porch                  6699
has_solar                  3910
dtype: uint64


In [99]:
count_types = [
    "RMB", "RMF", "RMH", "RMR", "RMT",
    "AP0", "AP1", "AP2", "AP3", "AP4",
    "UNT", "PRK",
    "FL1", "FL2", "FL3",
    "FPD", "FPM", "FPO", "FPW"
]

count_dup = (
    fixtures[
        fixtures["type"].isin(count_types)
    ]
    .groupby(["acct", "bld_num", "type"])
    .agg(
        rows=("units_num", "size"),
        sum_units=("units_num", "sum"),
        max_units=("units_num", "max"),
        min_units=("units_num", "min")
    )
    .reset_index()
)

print(
    count_dup[count_dup["rows"] > 1]
    .sort_values("rows", ascending=False)
    .head(50)
)

                  acct bld_num type  rows  sum_units  max_units  min_units
3049615  1155910140001       1  FL1    72       72.0        1.0        1.0
3049621  1155910140002       1  FL1    54       54.0        1.0        1.0
17791    0071560000004       1  AP1     4        0.0        0.0        0.0
103710   0230590000008       1  AP1     3        0.0        0.0        0.0
46       0020180000002       1  PRK     2     3154.0     1593.0     1561.0
5511     0041370000003       2  RMH     2        2.0        1.0        1.0
6718     0041770000226       1  RMB     2        4.0        2.0        2.0
6719     0041770000226       1  RMF     2        4.0        2.0        2.0
6720     0041770000226       1  RMT     2       10.0        5.0        5.0
13916    0070390100010       1  RMF     2        5.0        4.0        1.0
15716    0070550420013       1  RMR     2        7.0        4.0        3.0
28002    0102370001181       1  RMH     2        2.0        1.0        1.0
43877    0132700030001   

In [100]:
print(
    count_dup[count_dup["rows"] > 1]
    ["type"]
    .value_counts()
)

type
RMB    162
RMR    104
RMF     52
RMH     51
FPM     51
RMT     49
AP1     10
AP2     10
FPW     10
FPD      8
UNT      6
PRK      5
AP3      3
AP0      2
FL1      2
FPO      1
Name: count, dtype: int64[pyarrow]


In [101]:
print(
    count_dup[
        (count_dup["rows"] > 1)
        & count_dup["type"].isin(
            ["RMB", "RMF", "RMH", "RMT"]
        )
    ]
    .sort_values("rows", ascending=False)
    .head(30)
)

                 acct bld_num type  rows  sum_units  max_units  min_units
5511    0041370000003       2  RMH     2        2.0        1.0        1.0
6718    0041770000226       1  RMB     2        4.0        2.0        2.0
6719    0041770000226       1  RMF     2        4.0        2.0        2.0
6720    0041770000226       1  RMT     2       10.0        5.0        5.0
13916   0070390100010       1  RMF     2        5.0        4.0        1.0
28002   0102370001181       1  RMH     2        2.0        1.0        1.0
88198   0202260000012       2  RMT     2       10.0        5.0        5.0
115793  0251560000047       1  RMH     2        2.0        1.0        1.0
128340  0280360000006       1  RMT     2       11.0        8.0        3.0
131610  0282480010030       1  RMH     2        2.0        1.0        1.0
154157  0330940000002       1  RMB     2        4.0        3.0        1.0
178658  0350930460012       3  RMB     2        4.0        3.0        1.0
186840  0360390000006       1  RMB    

In [102]:
fixture_scalar_building = (
    fixtures[
        fixtures["type"].isin(
            ["STC", "STY", "INT", "WH", "WHT"]
        )
    ]
    .groupby(
        ["acct", "bld_num", "type"],
        as_index=False,
        sort=False
    )
    .agg(value=("units_num", "max"))
)

In [103]:
fixture_scalar_building = (
    fixture_scalar_building
    .pivot(
        index=["acct", "bld_num"],
        columns="type",
        values="value"
    )
    .reset_index()
)

In [104]:
fixture_scalar_building = fixture_scalar_building.rename(
    columns={
        "STC": "stories",
        "STY": "story_index",
        "INT": "interior_finish_pct",
        "WH": "wall_height",
        "WHT": "enclosure_wall_height"
    }
)

In [105]:
fixture_scalar_account = (
    fixture_scalar_building
    .groupby("acct", as_index=False, sort=False)
    .agg(
        building_count_fixture=("bld_num", "nunique"),

        max_stories=("stories", "max"),
        mean_stories=("stories", "mean"),

        mean_interior_finish_pct=("interior_finish_pct", "mean"),

        max_wall_height=("wall_height", "max"),
        mean_wall_height=("wall_height", "mean")
    )
)

In [106]:
count_types = [
    "RMB", "RMF", "RMH", "RMR", "RMT",
    "AP0", "AP1", "AP2", "AP3", "AP4",
    "UNT", "PRK",
    "FL1", "FL2", "FL3",
    "FPD", "FPM", "FPO", "FPW"
]

fixture_counts = (
    fixtures[
        fixtures["type"].isin(count_types)
    ]
    .groupby(["acct", "bld_num", "type"])
    .size()
    .reset_index(name="rows")
)

print(fixture_counts.shape)

print(
    fixture_counts[
        fixture_counts["rows"] > 1
    ]
    .sort_values("rows", ascending=False)
    .head(30)
)

(5304050, 4)
                  acct bld_num type  rows
3049615  1155910140001       1  FL1    72
3049621  1155910140002       1  FL1    54
17791    0071560000004       1  AP1     4
103710   0230590000008       1  AP1     3
46       0020180000002       1  PRK     2
5511     0041370000003       2  RMH     2
6718     0041770000226       1  RMB     2
6719     0041770000226       1  RMF     2
6720     0041770000226       1  RMT     2
13916    0070390100010       1  RMF     2
15716    0070550420013       1  RMR     2
28002    0102370001181       1  RMH     2
43877    0132700030001       1  AP0     2
43878    0132700030001       1  AP1     2
43879    0132700030001       1  AP2     2
4980599  1370560030013       1  RMR     2
211371   0393010100010       1  RMB     2
9        0010420000006       2  AP3     2
78771    0201100000020       1  FPD     2
88198    0202260000012       2  RMT     2
115793   0251560000047       1  RMH     2
128340   0280360000006       1  RMT     2
130598   028195003001

In [107]:
ROOM_TYPES = {
    "RMB": "bedrooms",
    "RMF": "full_baths",
    "RMH": "half_baths",
    "RMR": "rec_rooms",
    "RMT": "total_rooms",
}

rooms_building = (
    fixtures[
        fixtures["type"].isin(ROOM_TYPES.keys())
    ]
    .groupby(
        ["acct", "bld_num", "type"],
        as_index=False,
        sort=False
    )
    .agg(value=("units_num", "max"))
)

rooms_building = (
    rooms_building
    .pivot(
        index=["acct", "bld_num"],
        columns="type",
        values="value"
    )
    .reset_index()
    .rename(columns=ROOM_TYPES)
)

In [108]:
APARTMENT_TYPES = {
    "AP0": "efficiency_units",
    "AP1": "one_bedroom_units",
    "AP2": "two_bedroom_units",
    "AP3": "three_bedroom_units",
    "AP4": "four_bedroom_units",
}

apartment_building = (
    fixtures[
        fixtures["type"].isin(APARTMENT_TYPES.keys())
    ]
    .groupby(
        ["acct", "bld_num", "type"],
        as_index=False,
        sort=False
    )
    .agg(value=("units_num", "sum"))
)

apartment_building = (
    apartment_building
    .pivot(
        index=["acct", "bld_num"],
        columns="type",
        values="value"
    )
    .reset_index()
    .rename(columns=APARTMENT_TYPES)
)

In [109]:
stories_building = (
    fixtures[
        fixtures["type"] == "STC"
    ]
    .groupby(
        ["acct", "bld_num"],
        as_index=False,
        sort=False
    )
    .agg(stories=("units_num", "max"))
)

In [111]:
parking_building = (
    fixtures[
        fixtures["type"] == "PRK"
    ]
    .groupby(
        ["acct", "bld_num"],
        as_index=False,
        sort=False
    )
    .agg(parking_spaces=("units_num", "sum"))
)

In [112]:
FIREPLACE_TYPES = [
    "FL1", "FL2", "FL3",
    "FPD", "FPM", "FPO", "FPW"
]

fireplace_building = (
    fixtures[
        fixtures["type"].isin(FIREPLACE_TYPES)
    ]
    .groupby(
        ["acct", "bld_num"],
        as_index=False,
        sort=False
    )
    .agg(fireplace_count=("units_num", "sum"))
)

In [113]:
from functools import reduce

fixture_building_tables = [
    rooms_building,
    apartment_building,
    stories_building,
    parking_building,
    fireplace_building
]

fixtures_building = reduce(
    lambda left, right: left.merge(
        right,
        on=["acct", "bld_num"],
        how="outer",
        validate="one_to_one"
    ),
    fixture_building_tables
)

In [114]:
print(fixtures_building.shape)

print(
    "Duplicate account-building pairs:",
    fixtures_building[
        ["acct", "bld_num"]
    ].duplicated().sum()
)

(1258210, 15)
Duplicate account-building pairs: 0


In [115]:
fixtures_account = (
    fixtures_building
    .groupby("acct", as_index=False, sort=False)
    .agg(
        fixture_building_count=("bld_num", "nunique"),

        bedrooms=("bedrooms", "sum"),
        full_baths=("full_baths", "sum"),
        half_baths=("half_baths", "sum"),
        rec_rooms=("rec_rooms", "sum"),
        total_rooms=("total_rooms", "sum"),

        efficiency_units=("efficiency_units", "sum"),
        one_bedroom_units=("one_bedroom_units", "sum"),
        two_bedroom_units=("two_bedroom_units", "sum"),
        three_bedroom_units=("three_bedroom_units", "sum"),
        four_bedroom_units=("four_bedroom_units", "sum"),

        parking_spaces=("parking_spaces", "sum"),
        fireplace_count=("fireplace_count", "sum"),

        max_stories=("stories", "max"),
        mean_stories=("stories", "mean")
    )
)

In [116]:
print(fixtures_account.shape)

print(
    "Duplicate accounts:",
    fixtures_account["acct"].duplicated().sum()
)

print(
    fixtures_account[
        [
            "bedrooms",
            "full_baths",
            "half_baths",
            "total_rooms",
            "parking_spaces",
            "fireplace_count",
            "max_stories"
        ]
    ].describe(percentiles=[.5, .9, .95, .99])
)

(1210886, 16)
Duplicate accounts: 0
        bedrooms  full_baths  half_baths  total_rooms  parking_spaces  \
count  1210886.0   1210886.0   1210886.0    1210886.0       1210886.0   
mean     3.26155    2.045397    0.424177     6.685526         0.14688   
std     0.990049    0.791798    0.530352     2.008359       15.568639   
min          0.0         0.0         0.0          0.0             0.0   
50%          3.0         2.0         0.0          6.0             0.0   
90%          4.0         3.0         1.0          9.0             0.0   
95%          5.0         3.0         1.0         10.0             0.0   
99%          6.0         5.0         2.0         12.0             0.0   
max        105.0        56.0        24.0        244.0         10623.0   

       fireplace_count  max_stories  
count        1210886.0       9755.0  
mean          0.561585     2.946899  
std            0.62845     4.193128  
min                0.0          0.0  
50%                1.0          2.0  
90%  

In [118]:
sum_cols = [
    "bedrooms",
    "full_baths",
    "half_baths",
    "rec_rooms",
    "total_rooms",
    "efficiency_units",
    "one_bedroom_units",
    "two_bedroom_units",
    "three_bedroom_units",
    "four_bedroom_units",
    "parking_spaces",
    "fireplace_count"
]

g = fixtures_building.groupby(
    "acct",
    sort=False
)

# Fast built-in sum while preserving all-missing groups as NaN
fixture_sums = g[sum_cols].sum(min_count=1)

# Features that should not be summed
fixture_other = g.agg(
    fixture_building_count=("bld_num", "nunique"),
    max_stories=("stories", "max"),
    mean_stories=("stories", "mean")
)

# Combine
fixtures_account = (
    fixture_other
    .join(fixture_sums)
    .reset_index()
)

In [119]:
print(fixtures_account.shape)
print(
    "Duplicate accounts:",
    fixtures_account["acct"].duplicated().sum()
)

(1210886, 16)
Duplicate accounts: 0


In [120]:
print(
    fixtures_account[
        [
            "bedrooms",
            "full_baths",
            "half_baths",
            "total_rooms",
            "parking_spaces",
            "fireplace_count",
            "max_stories"
        ]
    ].describe(percentiles=[.5, .9, .95, .99])
)

        bedrooms  full_baths  half_baths  total_rooms  parking_spaces  \
count  1200767.0   1200717.0    495058.0    1200628.0           410.0   
mean    3.289035     2.06272    1.037515     6.742646      433.793659   
std     0.947659    0.772348    0.227229     1.919069      727.340576   
min          0.0         0.0         0.0          0.0             0.0   
50%          3.0         2.0         1.0          6.0           271.5   
90%          4.0         3.0         1.0          9.0          1084.0   
95%          5.0         3.0         1.0         10.0         1427.55   
99%          6.0         5.0         2.0         12.0         2621.03   
max        105.0        56.0        24.0        244.0         10623.0   

       fireplace_count  max_stories  
count         643641.0       9755.0  
mean          1.056513     2.946899  
std           0.469173     4.193128  
min                0.0          0.0  
50%                1.0          2.0  
90%                1.0          4.0  
95%

In [121]:
se1 = pd.read_csv(
    pdata2021 / "structural_elem1.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252",
    low_memory=False
)

se2 = pd.read_csv(
    pdata2021 / "structural_elem2.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252",
    low_memory=False
)

for df in [se1, se2]:
    for col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

print("SE1 shape:", se1.shape)
print("SE2 shape:", se2.shape)

print("\nSE1 columns:")
print(se1.columns.tolist())

print("\nSE2 columns:")
print(se2.columns.tolist())

SE1 shape: (8194794, 8)
SE2 shape: (1528914, 8)

SE1 columns:
['acct', 'bld_num', 'code', 'adj', 'type', 'type_dscr', 'category_dscr', 'dor_cd']

SE2 columns:
['acct', 'bld_num', 'code', 'adj', 'type', 'type_dscr', 'category_dscr', 'dor_cd']


In [122]:
print(se1.head(20))
print(se2.head(20))

             acct bld_num code         adj type            type_dscr  \
0   0020720000014       1    4    0.990000  CDU  Cond / Desir / Util   
1   0020720000014       1    4        <NA>  PCR   Physical Condition   
2   0020720000014       1    7    1.410000  GRD     Grade Adjustment   
3   0020720000014       1    3    8.000000  HAC         Heating / AC   
4   0020720000014       1    1  100.000000  XWR        Exterior Wall   
5   0020720000014       1    1    0.000000  FND      Foundation Type   
6   0020720000014       1   94    1.000000  CAD      Cost and Design   
7   0021440000001       1   94    0.800000  CAD      Cost and Design   
8   0021440000001       1    1    0.000000  FND      Foundation Type   
9   0021440000001       1    5  113.000000  XWR        Exterior Wall   
10  0021440000001       1    3    8.000000  HAC         Heating / AC   
11  0021440000001       1    9    1.170000  GRD     Grade Adjustment   
12  0021440000001       1    4        <NA>  PCR   Physical Condi

In [123]:
print(
    se1["type"]
    .value_counts()
    .head(40)
)

print(
    se2["type"]
    .value_counts()
    .head(40)
)

type
XWR    1780247
FND    1229878
GRD    1229234
CDU    1229225
PCR    1229225
HAC    1229202
CAD     267468
PC          44
XWC         33
HTG         32
PAR         31
CLG         31
FN          31
EO          31
PLM         30
SPR         27
MI          21
CON          4
Name: count, dtype: int64[pyarrow]
type
XWC     154378
PLM     150076
PAR     149961
HTG     149634
CLG     149531
FN      148717
PC      148598
EO      148453
SPR     147396
MI      145173
MA       22012
CON       7649
NHF       3350
GRD        954
HAC        882
XWR        439
PCR        388
ESTR       301
FND        236
CDU        221
EL         204
LEED       197
SL         146
CAD         18
Name: count, dtype: int64[pyarrow]


In [124]:
se1_types = ["CDU", "PCR", "GRD", "HAC", "FND", "CAD"]

se1_core = se1.loc[
    se1["type"].isin(se1_types),
    [
        "acct",
        "bld_num",
        "type",
        "code",
        "adj",
        "category_dscr"
    ]
].drop_duplicates()

dup1 = se1_core.duplicated(
    ["acct", "bld_num", "type"],
    keep=False
)

print("SE1 conflicting/repeated core rows:")
print(
    se1_core.loc[dup1, "type"]
    .value_counts()
)

SE1 conflicting/repeated core rows:
type
FND    1327
HAC      22
CAD       8
Name: count, dtype: int64[pyarrow]


In [125]:
se2_types = [
    "PC",
    "FN",
    "CLG",
    "HTG",
    "PLM",
    "PAR",
    "SPR",
    "EO",
    "CON"
]

se2_core = se2.loc[
    se2["type"].isin(se2_types),
    [
        "acct",
        "bld_num",
        "type",
        "code",
        "adj",
        "category_dscr"
    ]
].drop_duplicates()

dup2 = se2_core.duplicated(
    ["acct", "bld_num", "type"],
    keep=False
)

print("\nSE2 conflicting/repeated core rows:")
print(
    se2_core.loc[dup2, "type"]
    .value_counts()
)


SE2 conflicting/repeated core rows:
type
HTG    3856
CLG    3577
PLM    3330
PAR    3218
SPR     602
PC        6
CON       2
FN        2
Name: count, dtype: int64[pyarrow]


In [127]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# COMBINE STRUCTURAL FILES
# --------------------------------------------------

structural_all = pd.concat(
    [
        se1[
            [
                "acct", "bld_num", "code", "adj",
                "type", "type_dscr", "category_dscr", "dor_cd"
            ]
        ],
        se2[
            [
                "acct", "bld_num", "code", "adj",
                "type", "type_dscr", "category_dscr", "dor_cd"
            ]
        ]
    ],
    ignore_index=True
).drop_duplicates()

structural_all["adj_num"] = pd.to_numeric(
    structural_all["adj"],
    errors="coerce"
)

structural_all["category_upper"] = (
    structural_all["category_dscr"]
    .fillna("")
    .str.upper()
)

In [128]:
structural_parts = []

# Grade
grade = (
    structural_all[
        structural_all["type"] == "GRD"
    ]
    .groupby("acct")["adj_num"]
    .agg(["mean", "min", "max"])
    .rename(
        columns={
            "mean": "mean_grade_adj",
            "min": "min_grade_adj",
            "max": "max_grade_adj"
        }
    )
)

structural_parts.append(grade)

# CDU
cdu = (
    structural_all[
        structural_all["type"] == "CDU"
    ]
    .groupby("acct")["adj_num"]
    .agg(["mean", "min", "max"])
    .rename(
        columns={
            "mean": "mean_cdu_adj",
            "min": "min_cdu_adj",
            "max": "max_cdu_adj"
        }
    )
)

structural_parts.append(cdu)

In [129]:
structural_parts = []

# Grade
grade = (
    structural_all[
        structural_all["type"] == "GRD"
    ]
    .groupby("acct")["adj_num"]
    .agg(["mean", "min", "max"])
    .rename(
        columns={
            "mean": "mean_grade_adj",
            "min": "min_grade_adj",
            "max": "max_grade_adj"
        }
    )
)

structural_parts.append(grade)

# CDU
cdu = (
    structural_all[
        structural_all["type"] == "CDU"
    ]
    .groupby("acct")["adj_num"]
    .agg(["mean", "min", "max"])
    .rename(
        columns={
            "mean": "mean_cdu_adj",
            "min": "min_cdu_adj",
            "max": "max_cdu_adj"
        }
    )
)

structural_parts.append(cdu)

In [130]:
condition = structural_all[
    structural_all["type"].isin(["PCR", "PC"])
][
    ["acct", "category_upper"]
].drop_duplicates()

condition["condition_score"] = np.nan

condition.loc[
    condition["category_upper"].str.contains("POOR"),
    "condition_score"
] = 1

condition.loc[
    condition["category_upper"].str.contains("FAIR"),
    "condition_score"
] = 2

condition.loc[
    condition["category_upper"].str.contains(
        "AVERAGE|AVG|NORMAL",
        regex=True
    ),
    "condition_score"
] = 3

condition.loc[
    condition["category_upper"].str.contains("GOOD")
    & ~condition["category_upper"].str.contains("VERY GOOD"),
    "condition_score"
] = 4

condition.loc[
    condition["category_upper"].str.contains("VERY GOOD"),
    "condition_score"
] = 5

condition.loc[
    condition["category_upper"].str.contains("EXCELLENT"),
    "condition_score"
] = 6

In [131]:
condition_account = (
    condition
    .groupby("acct")["condition_score"]
    .agg(["mean", "min", "max"])
    .rename(
        columns={
            "mean": "mean_condition_score",
            "min": "worst_condition_score",
            "max": "best_condition_score"
        }
    )
)

structural_parts.append(condition_account)

In [132]:
foundation = structural_all[
    structural_all["type"] == "FND"
].copy()

foundation_account = (
    foundation
    .groupby("acct")
    .agg(
        foundation_type_count=("code", "nunique")
    )
)

foundation_flags = (
    foundation
    .assign(
        has_slab_foundation=(
            foundation["category_upper"]
            .str.contains("SLAB", regex=False)
        ).astype("uint8"),

        has_pier_foundation=(
            foundation["category_upper"]
            .str.contains("PIER", regex=False)
        ).astype("uint8")
    )
    .groupby("acct")[
        [
            "has_slab_foundation",
            "has_pier_foundation"
        ]
    ]
    .max()
)

foundation_account = foundation_account.join(
    foundation_flags,
    how="left"
)

structural_parts.append(foundation_account)

In [133]:
hvac = structural_all[
    structural_all["type"].isin(
        ["HAC", "HTG", "CLG"]
    )
].copy()

hvac["has_central_hvac"] = (
    hvac["category_upper"]
    .str.contains("CENTRAL", regex=False)
).astype("uint8")

hvac_account = (
    hvac
    .groupby("acct")
    .agg(
        hvac_type_count=("code", "nunique"),
        has_central_hvac=("has_central_hvac", "max")
    )
)

structural_parts.append(hvac_account)

In [134]:
walls = structural_all[
    structural_all["type"].isin(
        ["XWR", "XWC"]
    )
].copy()

walls["has_brick_stone"] = (
    walls["category_upper"]
    .str.contains("BRICK|STONE", regex=True)
).astype("uint8")

walls["has_frame_exterior"] = (
    walls["category_upper"]
    .str.contains("FRAME", regex=False)
).astype("uint8")

walls["has_stucco"] = (
    walls["category_upper"]
    .str.contains("STUCCO", regex=False)
).astype("uint8")

walls["has_concrete_exterior"] = (
    walls["category_upper"]
    .str.contains("CONCR", regex=False)
).astype("uint8")

wall_account = (
    walls
    .groupby("acct")
    .agg(
        exterior_wall_type_count=("code", "nunique"),
        has_brick_stone=("has_brick_stone", "max"),
        has_frame_exterior=("has_frame_exterior", "max"),
        has_stucco=("has_stucco", "max"),
        has_concrete_exterior=("has_concrete_exterior", "max")
    )
)

structural_parts.append(wall_account)

In [135]:
structural_summary = (
    structural_all
    .groupby("acct")
    .agg(
        structural_building_count=("bld_num", "nunique"),
        structural_element_types=("type", "nunique"),
        structural_record_count=("type", "size")
    )
)

structural_parts.append(structural_summary)

In [136]:
structural_account = (
    pd.concat(
        structural_parts,
        axis=1
    )
    .reset_index()
)

print(structural_account.shape)

print(
    "Duplicate structural accounts:",
    structural_account["acct"].duplicated().sum()
)

(1267836, 23)
Duplicate structural accounts: 0


In [138]:
property_2021 = (
    real
    .merge(
        rental_building_evidence,
        on="acct",
        how="left",
        validate="one_to_one"
    )
    .merge(
        exemption_evidence,
        on="acct",
        how="left",
        validate="one_to_one"
    )
    .merge(
        owner_evidence_simple,
        on="acct",
        how="left",
        validate="one_to_one"
    )
)

In [139]:
binary_cols = [
    "explicit_rental",
    "explicit_rental_building",
    "explicit_rental_state",
    "explicit_rental_pgi",
    "has_res_homestead",
    "has_partial_homestead",
    "has_apportioned_residential",
    "has_low_income_housing",
    "any_homestead_evidence",
    "has_corporate_owner"
]

for col in binary_cols:
    if col in property_2021.columns:
        property_2021[col] = (
            property_2021[col]
            .fillna(0)
            .astype("uint8")
        )

In [140]:
print(property_2021.shape)
print("Unique accounts:", property_2021["acct"].nunique())
print("Duplicates:", property_2021["acct"].duplicated().sum())

(1507914, 38)
Unique accounts: 1507914
Duplicates: 0


In [141]:
real = pd.read_csv(
    pdata2021 / "real_acct.txt",
    sep="\t",
    dtype=str,
    encoding="cp1252",
    low_memory=False
)

for col in real.columns:
    real[col] = (
        real[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

print(real.shape)
print("Duplicates:", real["acct"].duplicated().sum())

(1507914, 71)
Duplicates: 0


In [142]:
property_features_2021 = (
    property_2021
    .merge(
        fixtures_account,
        on="acct",
        how="left",
        validate="one_to_one"
    )
    .merge(
        extra_features_account,
        on="acct",
        how="left",
        validate="one_to_one"
    )
    .merge(
        structural_account,
        on="acct",
        how="left",
        validate="one_to_one"
    )
)

In [143]:
print(property_features_2021.shape)
print(
    "Duplicate accounts:",
    property_features_2021["acct"].duplicated().sum()
)

(1507914, 85)
Duplicate accounts: 0


In [144]:
property_features_2021.to_parquet(
    "property_features_2021_checkpoint.parquet",
    index=False
)

In [145]:
check_cols = [
    "explicit_rental",
    "any_homestead_evidence",
    "has_corporate_owner",
    "likely_rental",
    "estimated_units_prelim",
    "bedrooms",
    "full_baths",
    "total_rooms",
    "has_pool",
    "has_garage",
    "has_cracked_slab",
    "mean_grade_adj",
    "mean_cdu_adj",
    "mean_condition_score"
]

for col in check_cols:
    print(f"{col}: {col in property_features_2021.columns}")

explicit_rental: True
any_homestead_evidence: True
has_corporate_owner: True
likely_rental: False
estimated_units_prelim: False
bedrooms: True
full_baths: True
total_rooms: True
has_pool: True
has_garage: True
has_cracked_slab: True
mean_grade_adj: True
mean_cdu_adj: True
mean_condition_score: True


In [147]:
property_features_2021.to_parquet(
    "property_features_2021_checkpoint.parquet",
    index=False
)

In [148]:
print(property_features_2021.columns.tolist())

['acct', 'yr', 'mailto', 'mail_addr_1', 'mail_addr_2', 'mail_city', 'mail_state', 'mail_zip', 'site_addr_1', 'site_addr_2', 'site_addr_3', 'state_class', 'Neighborhood_Code', 'Neighborhood_Grp', 'Market_Area_1', 'Market_Area_1_Dscr', 'yr_impr', 'bld_ar', 'land_ar', 'acreage', 'land_val', 'bld_val', 'tot_appr_val', 'tot_mkt_val', 'new_own_dt', 'explicit_rental', 'explicit_rental_building', 'explicit_rental_state', 'explicit_rental_pgi', 'in_building_res', 'in_building_other', 'exempt_cat', 'has_res_homestead', 'has_partial_homestead', 'has_apportioned_residential', 'has_low_income_housing', 'any_homestead_evidence', 'has_corporate_owner', 'fixture_building_count', 'max_stories', 'mean_stories', 'bedrooms', 'full_baths', 'half_baths', 'rec_rooms', 'total_rooms', 'efficiency_units', 'one_bedroom_units', 'two_bedroom_units', 'three_bedroom_units', 'four_bedroom_units', 'parking_spaces', 'fireplace_count', 'has_pool', 'has_pool_spa', 'has_garage', 'has_carport', 'has_shed', 'has_porch', 'ha

In [149]:
feature_cols = [
    "bedrooms",
    "full_baths",
    "half_baths",
    "total_rooms",
    "fireplace_count",
    "parking_spaces",
    "max_stories",

    "has_pool",
    "has_pool_spa",
    "has_garage",
    "has_carport",
    "has_shed",
    "has_porch",
    "has_outdoor_kitchen",
    "has_solar",
    "has_foundation_repair",
    "has_cracked_slab",

    "mean_grade_adj",
    "mean_cdu_adj",
    "mean_condition_score",
    "foundation_type_count",
    "has_slab_foundation",
    "has_pier_foundation",
    "has_central_hvac",
    "exterior_wall_type_count"
]

coverage = (
    property_features_2021[feature_cols]
    .notna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print(coverage)

exterior_wall_type_count    84.07
has_central_hvac            84.07
mean_condition_score        83.95
mean_grade_adj              79.65
has_slab_foundation         79.63
full_baths                  79.63
mean_cdu_adj                79.63
has_pier_foundation         79.63
bedrooms                    79.63
foundation_type_count       79.63
total_rooms                 79.62
fireplace_count             42.68
has_cracked_slab            35.72
has_pool_spa                35.72
has_foundation_repair       35.72
has_solar                   35.72
has_pool                    35.72
has_porch                   35.72
has_shed                    35.72
has_carport                 35.72
has_outdoor_kitchen         35.72
has_garage                  35.72
half_baths                  32.83
max_stories                  0.65
parking_spaces               0.03
dtype: float64


In [150]:
property_feature_cols = [
    "exterior_wall_type_count",
    "has_central_hvac",
    "mean_condition_score",
    "mean_grade_adj",
    "has_slab_foundation",
    "full_baths",
    "mean_cdu_adj",
    "has_pier_foundation",
    "bedrooms",
    "foundation_type_count",
    "total_rooms",
    "fireplace_count",
    "has_cracked_slab",
    "has_pool_spa",
    "has_foundation_repair",
    "has_solar",
    "has_pool",
    "has_porch",
    "has_shed",
    "has_carport",
    "has_outdoor_kitchen",
    "has_garage",
    "half_baths"
]

In [151]:
extra_binary_cols = [
    "has_cracked_slab",
    "has_pool_spa",
    "has_foundation_repair",
    "has_solar",
    "has_pool",
    "has_porch",
    "has_shed",
    "has_carport",
    "has_outdoor_kitchen",
    "has_garage"
]

property_features_2021[extra_binary_cols] = (
    property_features_2021[extra_binary_cols]
    .fillna(0)
    .astype("uint8")
)

In [152]:
property_features_2021["fixture_data_available"] = (
    property_features_2021["acct"]
    .isin(fixtures_account["acct"])
    .astype("uint8")
)

In [153]:
mask = property_features_2021["fixture_data_available"] == 1

for col in [
    "half_baths",
    "fireplace_count",
    "rec_rooms"
]:
    property_features_2021.loc[
        mask & property_features_2021[col].isna(),
        col
    ] = 0

In [154]:
missing_indicator_cols = [
    "exterior_wall_type_count",
    "has_central_hvac",
    "mean_condition_score",
    "mean_grade_adj",
    "full_baths",
    "mean_cdu_adj",
    "bedrooms",
    "foundation_type_count",
    "total_rooms",
    "fireplace_count",
    "half_baths",
    "max_stories",
    "parking_spaces"
]

for col in missing_indicator_cols:
    property_features_2021[f"{col}_missing"] = (
        property_features_2021[col]
        .isna()
        .astype("uint8")
    )

In [155]:
property_features_2021["bathroom_equivalent"] = (
    property_features_2021["full_baths"]
    + 0.5 * property_features_2021["half_baths"].fillna(0)
)

In [156]:
property_features_2021["has_structural_issue"] = (
    (
        property_features_2021["has_cracked_slab"].eq(1)
        | property_features_2021["has_foundation_repair"].eq(1)
    )
    .astype("uint8")
)

In [157]:
property_features_2021["has_major_amenity"] = (
    property_features_2021[
        [
            "has_pool",
            "has_pool_spa",
            "has_garage",
            "has_carport",
            "has_outdoor_kitchen"
        ]
    ]
    .max(axis=1)
    .astype("uint8")
)

In [158]:
real_numeric_cols = [
    "yr_impr",
    "bld_ar",
    "land_ar",
    "acreage",
    "land_val",
    "bld_val",
    "tot_appr_val",
    "tot_mkt_val"
]

for col in real_numeric_cols:
    if col in property_features_2021.columns:
        property_features_2021[f"{col}_num"] = pd.to_numeric(
            property_features_2021[col],
            errors="coerce"
        )

In [159]:
property_features_2021["property_age"] = (
    2021 - property_features_2021["yr_impr_num"]
)

property_features_2021.loc[
    (property_features_2021["property_age"] < 0)
    | (property_features_2021["property_age"] > 250),
    "property_age"
] = np.nan

In [160]:
property_features_2021["market_value_per_bld_sqft"] = (
    property_features_2021["tot_mkt_val_num"]
    / property_features_2021["bld_ar_num"]
)

property_features_2021.loc[
    property_features_2021["bld_ar_num"] <= 0,
    "market_value_per_bld_sqft"
] = np.nan

In [161]:
print(
    "estimated_units_prelim" in property_features_2021.columns
)

False


In [162]:
property_features_2021 = property_features_2021.merge(
    unit_check[
        [
            "acct",
            "max_reported_units",
            "sum_building_units",
            "units_from_type",
            "estimated_units_prelim"
        ]
    ],
    on="acct",
    how="left",
    validate="one_to_one"
)

In [163]:
valid_units = (
    property_features_2021["estimated_units_prelim"] > 0
)

property_features_2021.loc[
    valid_units,
    "market_value_per_unit"
] = (
    property_features_2021.loc[
        valid_units,
        "tot_mkt_val_num"
    ]
    /
    property_features_2021.loc[
        valid_units,
        "estimated_units_prelim"
    ]
)

In [164]:
valid_units = (
    property_features_2021["estimated_units_prelim"] > 0
)

property_features_2021.loc[
    valid_units,
    "market_value_per_unit"
] = (
    property_features_2021.loc[
        valid_units,
        "tot_mkt_val_num"
    ]
    /
    property_features_2021.loc[
        valid_units,
        "estimated_units_prelim"
    ]
)

In [165]:
print("Rows:", len(property_features_2021))
print("Columns:", property_features_2021.shape[1])
print("Unique accounts:", property_features_2021["acct"].nunique())
print("Duplicates:", property_features_2021["acct"].duplicated().sum())

Rows: 1507914
Columns: 117
Unique accounts: 1507914
Duplicates: 0


In [166]:
property_features_2021.to_parquet(
    "property_features_2021_checkpoint.parquet",
    index=False
)